# Weather ETL Pipeline

A simple ETL (Extract, Transform, Load) pipeline that pulls live weather data from the OpenWeather API for multiple cities, cleans and structures it with Pandas, and stores it as a CSV for analysis.

Built as part of **Week 7 — AnalystLab Africa Data Analytics Internship** (#AnalystLabAfrica).

**Cities analyzed:** Nairobi, London, New York
**Pipeline stages:** Extract → Transform → Load → Analyze


### Install dependencies

We install the three libraries this project depends on: `requests` for calling the API, `pandas` for structuring the data, and `python-dotenv` for securely loading the API key.


In [1]:
pip install requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load the API key securely

We load the OpenWeather API key from a `.env` file using `python-dotenv`, instead of hardcoding it in the notebook. This keeps the key out of version control (it's excluded via `.gitignore`) so it's never exposed on GitHub.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

print("Key loaded:", API_KEY is not None)  # should print True

Key loaded: True


### Test API call — Nairobi

Before looping through multiple cities, we make a single test request to the OpenWeather `/data/2.5/weather` endpoint for Nairobi. This lets us inspect the raw JSON response and confirm the request works before building out the full extraction logic.

We pass `units=metric` so temperature comes back in °C rather than the API's default Kelvin.


In [3]:
import requests

city = "Nairobi"
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"   # gives temp in °C instead of Kelvin
}

response = requests.get(url, params=params)
print(response.status_code)
data = response.json()
data

200


{'coord': {'lon': 36.8167, 'lat': -1.2833},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 17.93,
  'feels_like': 17.66,
  'temp_min': 17.62,
  'temp_max': 17.93,
  'pressure': 1019,
  'humidity': 72,
  'sea_level': 1019,
  'grnd_level': 844},
 'visibility': 10000,
 'wind': {'speed': 5.14, 'deg': 130},
 'clouds': {'all': 56},
 'dt': 1788251870,
 'sys': {'type': 1,
  'id': 2558,
  'country': 'KE',
  'sunrise': 1788233428,
  'sunset': 1788276940},
 'timezone': 10800,
 'id': 184745,
 'name': 'Nairobi',
 'cod': 200}

### Build the extraction function

The raw API response is deeply nested (e.g. temperature sits inside `data['main']['temp']`). We write an `extract_weather()` function that pulls out only the fields we need — city, temperature, humidity, condition, wind speed, and datetime — and flattens them into a single clean dictionary. The API's Unix timestamp is also converted into a readable Python `datetime` object.


In [4]:
from datetime import datetime

def extract_weather(data):
    return {
        "city": data["name"],
        "temperature_c": data["main"]["temp"],
        "humidity_pct": data["main"]["humidity"],
        "condition": data["weather"][0]["description"],
        "wind_speed_mps": data["wind"]["speed"],
        "datetime": datetime.fromtimestamp(data["dt"])
    }

nairobi_weather = extract_weather(data)
nairobi_weather

{'city': 'Nairobi',
 'temperature_c': 17.93,
 'humidity_pct': 72,
 'condition': 'broken clouds',
 'wind_speed_mps': 5.14,
 'datetime': datetime.datetime(2026, 9, 1, 11, 37, 50)}

### Loop over all cities

We repeat the same request-and-extract process for all 3 target cities (Nairobi, London, New York), appending each result to a `weather_records` list. A success/failure print statement is added per city, so if one request fails, it's visible immediately rather than silently breaking the pipeline.


In [5]:
cities = ["Nairobi", "London", "New York"]

weather_records = []

for city in cities:
    params = {
        "q" : city,
        "appid" : API_KEY,
        "units" : "metric"
    }
    response = requests.get(url, params=params)

    if response.status_code == 200:
        city_data = response.json()
        weather_records.append(extract_weather(city_data))
        print(f"{city} fetched successfully")
    else:
        print(f"Failed to fetch {city} - status code: {response.status_code}")

weather_records

Nairobi fetched successfully
London fetched successfully
New York fetched successfully


[{'city': 'Nairobi',
  'temperature_c': 17.93,
  'humidity_pct': 72,
  'condition': 'broken clouds',
  'wind_speed_mps': 5.14,
  'datetime': datetime.datetime(2026, 9, 1, 11, 37, 50)},
 {'city': 'London',
  'temperature_c': 16.8,
  'humidity_pct': 71,
  'condition': 'broken clouds',
  'wind_speed_mps': 2.57,
  'datetime': datetime.datetime(2026, 9, 1, 11, 37, 39)},
 {'city': 'New York',
  'temperature_c': 22.24,
  'humidity_pct': 92,
  'condition': 'overcast clouds',
  'wind_speed_mps': 2.68,
  'datetime': datetime.datetime(2026, 9, 1, 11, 37, 16)}]

## Task 1: Extract — Summary

**Goal:** Pull live weather data for 3 cities from the OpenWeather API.

### What we did

1. **Secured the API key**
   - Stored the key in a `.env` file (not hardcoded in the notebook).
   - Loaded it with `python-dotenv` so it never appears in the code itself.
   - This matters because a public GitHub repo with a raw API key gets scraped and abused — `.env` is excluded via `.gitignore`.

2. **Made a single test call**
   - Hit OpenWeather's `/data/2.5/weather` endpoint for one city (Nairobi) to inspect the raw JSON response before building anything bigger.
   - Used `units=metric` in the request so temperature comes back in °C rather than Kelvin.

3. **Mapped the JSON structure**
   - The raw response is deeply nested (e.g. temperature is inside `data['main']['temp']`, condition is inside a list `data['weather'][0]['description']`).
   - Wrote an `extract_weather()` function that pulls out just the fields we need and flattens them into one simple dictionary per city:
     - `city`, `temperature_c`, `humidity_pct`, `condition`, `wind_speed_mps`, `datetime`
   - Converted the API's Unix timestamp (`dt`) into a readable Python `datetime` object.

4. **Looped over all 3 cities**
   - Nairobi, London, New York.
   - Each city's data is fetched, extracted, and appended to a list called `weather_records`.
   - Added success/failure print statements so a failed request for one city doesn't silently break the pipeline — we'd see it immediately.

### Output

A list of 3 clean dictionaries — one per city — ready to be converted into a structured table.

### Why it matters

This is the **Extract** step of ETL: getting raw data out of a source system (an API here) reliably, with visibility into what succeeded or failed, before any cleaning or analysis happens.

### Convert to a DataFrame

`weather_records` — our list of dictionaries — is loaded into a Pandas DataFrame, giving us one row per city and one column per field.


In [6]:
import pandas as pd
df = pd.DataFrame(weather_records)
df

,city,temperature_c,humidity_pct,condition,wind_speed_mps,datetime
0,Nairobi,17.93,72,broken clouds,5.14,2026-09-01 11:37:50
1,London,16.80,71,broken clouds,2.57,2026-09-01 11:37:39
2,New York,22.24,92,overcast clouds,2.68,2026-09-01 11:37:16


### Check data types

Before renaming anything, we check the DataFrame's dtypes to confirm temperature and wind speed came through as floats, humidity as an integer, and datetime as a true `datetime64` type rather than plain text.


In [7]:
df.dtypes

city                      object
temperature_c            float64
humidity_pct               int64
condition                 object
wind_speed_mps           float64
datetime          datetime64[ns]
dtype: object

### Rename columns for readability

We relabel the technical field names into clear, human-readable column headers with units included, e.g. `Temperature(°C)`, `Wind Speed (m/s)`.


In [8]:
df = df.rename(columns={
    "temperature_c" : "Temperature(°C)",
    "humidity_pct" : "Humidity(%)",
    "wind_speed_mps" : "Wind Speed (m/s)",
    "city" : "City",
    "condition" : "Weather Condition",
    "datetime" : "Date & Time"
})

df

,City,Temperature(°C),Humidity(%),Weather Condition,Wind Speed (m/s),Date & Time
0,Nairobi,17.93,72,broken clouds,5.14,2026-09-01 11:37:50
1,London,16.80,71,broken clouds,2.57,2026-09-01 11:37:39
2,New York,22.24,92,overcast clouds,2.68,2026-09-01 11:37:16


## Task 2: Transform — Summary

**Goal:** Turn the raw list of dictionaries into a clean, well-structured dataset.

### What we did

1. **Converted to a DataFrame**
   - Loaded `weather_records` into a Pandas DataFrame — one row per city, one column per field.

2. **Checked data types**
   - Confirmed temperature and wind speed came through as floats, humidity as an integer, and datetime as a true `datetime64` type rather than plain text — important for any later sorting, filtering, or plotting.

3. **Renamed columns for readability**
   - Relabeled technical field names (`temperature_c`, `humidity_pct`, etc.) into human-readable column headers with units included, e.g. `Temperature(°C)`, `Wind Speed (m/s)`.

### Why it matters

This is the **Transform** step of ETL: taking raw extracted data and reshaping it into something structured, correctly typed, and readable — ready to be stored or analyzed.


### Save to CSV

The cleaned DataFrame is saved to `data/weather_data.csv`, with the row index excluded (`index=False`) to keep the file tidy.


In [9]:
import os
# Make sure the data folder exists
os.makedirs("data", exist_ok=True)

df.to_csv("data/weather_data.csv", index=False)
print("Data saved to data/weather data.csv")

Data saved to data/weather data.csv


## Task 3: Load — Summary

**Goal:** Store the cleaned dataset somewhere it can be reused for analysis.

### What we did

- Saved the transformed DataFrame to `data/weather_data.csv`, with the row index excluded (`index=False`) to keep the file clean.
- Chose CSV as the storage format for this project because it's simple, portable, and easy to reload for future analysis or to open directly in Excel/Sheets.

### Why it matters

This is the **Load** step of ETL: persisting the cleaned data so it's available for downstream use, without needing to re-run the API calls every time.


### Confirm final column layout

A quick check of the final column names, confirming the DataFrame is clean and ready for analysis.


In [10]:
df.columns.tolist()

['City',
 'Temperature(°C)',
 'Humidity(%)',
 'Weather Condition',
 'Wind Speed (m/s)',
 'Date & Time']

### Run comparative analysis

We compare the 3 cities directly: which is warmest/coolest, which has the highest humidity, and how weather conditions differ across all three.


In [11]:
# Which city is warmest / coolest?
warmest = df.loc[df["Temperature(°C)"].idxmax()]
coolest = df.loc[df["Temperature(°C)"].idxmin()]

print(f"Warmest city: {warmest['City']} at {warmest['Temperature(°C)']}°C")
print(f"Coolest city: {coolest['City']} at {coolest['Temperature(°C)']}°C")

# Which city is most humid?
most_humid = df.loc[df["Humidity(%)"].idxmax()]
print(f"💧 Most humid city: {most_humid['City']} at {most_humid['Humidity(%)']}%")

# Weather conditions side by side
print("\n Weather conditions:")
print(df[["City", "Weather Condition"]].to_string(index=False))

Warmest city: New York at 22.24°C
Coolest city: London at 16.8°C
💧 Most humid city: New York at 92%

 Weather conditions:
    City Weather Condition
 Nairobi     broken clouds
  London     broken clouds
New York   overcast clouds


## Task 4: Findings Summary

On 27 August 2026, weather was compared across three cities: Nairobi, London, and New York.

- **New York** was the warmest of the three at 22.3°C, while **London** was the coolest at 19.16°C — a gap of just over 3°C despite the very different geography.
- **London** also had the highest humidity at 92%, notably higher than Nairobi (56%) and New York (85%), which lines up with its overcast sky conditions.
- Conditions varied across all three: Nairobi had few clouds, London was overcast, and New York had scattered clouds — none of the three cities recorded clear skies or rain at the time of the snapshot.
- Interestingly, Nairobi's tropical highland climate produced the lowest humidity despite not being the coolest city, showing that temperature and humidity don't always move together.